# Cell 1 — Install and imports

In [1]:
!pip -q install ultralytics pyyaml opencv-python-headless

import os
import gc
import json
import shutil
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO

print("Imports done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports done


# Cell 2 — Resolve stage1 bundle input

In [2]:
STAGE1_ROOT = "/kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle"

assert os.path.exists(STAGE1_ROOT), f"Stage1 bundle not found: {STAGE1_ROOT}"
assert os.path.exists(os.path.join(STAGE1_ROOT, "stage1_summary.json")), "stage1_summary.json missing"
assert os.path.exists(os.path.join(STAGE1_ROOT, "train_records.csv")), "train_records.csv missing"
assert os.path.exists(os.path.join(STAGE1_ROOT, "val_records.csv")), "val_records.csv missing"

with open(os.path.join(STAGE1_ROOT, "stage1_summary.json"), "r") as f:
    stage1_summary = json.load(f)

print("STAGE1_ROOT:", STAGE1_ROOT)
print(json.dumps(stage1_summary, indent=2))

STAGE1_ROOT: /kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle
{
  "PREP_ROOT": "/kaggle/input/datasets/naresh26032005/vinbigdata-prep",
  "COMP_ROOT": "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection",
  "DATA_1024": "/kaggle/working/yolo1024_stage1_bundle/data/yolo1024",
  "stage1_export": "/kaggle/working/yolo1024_stage1_bundle/yolov8m_1024_stage1.pt",
  "train_images": 12000,
  "val_images": 3000,
  "train_records": 18280,
  "val_records": 3000,
  "focus_classes": [
    1,
    2,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13
  ],
  "rare_classes": [
    2,
    8,
    9,
    11,
    12,
    13
  ]
}


# Cell 3 — Config

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = 0 if torch.cuda.is_available() else "cpu"

YOLO_SIZE = 1024
MODEL_NAME = os.path.join(STAGE1_ROOT, "yolov8m_1024_stage1.pt")

# Keep stage2 short to avoid timeout
EPOCHS_STAGE2 = 5

BATCH = 4
WORKERS = 2

CLASS_NAMES = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
]

NO_FINDING_ID = 14
FOCUS_CLASS_IDS = {1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13}
RARE_CLASS_IDS = {2, 8, 9, 11, 12, 13}

WORK_DIR = "/kaggle/working/yolo1024_stage2_bundle"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Work dir:", WORK_DIR)

Device: 0
Work dir: /kaggle/working/yolo1024_stage2_bundle


# Cell 4 — Load stage1 records

In [4]:
train_records_df = pd.read_csv(os.path.join(STAGE1_ROOT, "train_records.csv"))
val_records_df = pd.read_csv(os.path.join(STAGE1_ROOT, "val_records.csv"))

print("train records:", len(train_records_df))
print("val records:", len(val_records_df))
print(train_records_df.head())

train records: 18280
val records: 3000
                           image_id       kind  \
0  000434271f63a053c4128a0ba6352c7f       full   
1  0005e8e3701dfb1dd93d53e2ff537b6e       full   
2  0005e8e3701dfb1dd93d53e2ff537b6e        roi   
3  0005e8e3701dfb1dd93d53e2ff537b6e  roi_tight   
4  0006e0a85696f6bb578e84fafa9a5607       full   

                                        img_rel_path  \
0  data/yolo1024/images/train/full/000434271f63a0...   
1  data/yolo1024/images/train/full/0005e8e3701dfb...   
2  data/yolo1024/images/train/roi/0005e8e3701dfb1...   
3  data/yolo1024/images/train/roi_tight/0005e8e37...   
4  data/yolo1024/images/train/full/0006e0a85696f6...   

                                      label_rel_path  has_abnormal  \
0  data/yolo1024/labels/train/full/000434271f63a0...             0   
1  data/yolo1024/labels/train/full/0005e8e3701dfb...             1   
2  data/yolo1024/labels/train/roi/0005e8e3701dfb1...             1   
3  data/yolo1024/labels/train/roi_tight/000

# Cell 5 — Helpers

In [5]:
def rel_to_abs(rel_path):
    return os.path.join(STAGE1_ROOT, rel_path)

def parse_classes(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [int(x) for x in str(s).split() if str(x).strip().isdigit()]

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

train_freq = Counter()
for s in train_records_df[train_records_df["kind"] == "full"]["classes"].tolist():
    for c in parse_classes(s):
        if c != NO_FINDING_ID:
            train_freq[c] += 1

abn_freq_values = [v for k, v in train_freq.items() if v > 0]
base_weight = float(np.median(abn_freq_values)) if len(abn_freq_values) else 1.0

def repeat_base(classes):
    abn = [c for c in classes if c != NO_FINDING_ID]
    if len(abn) == 0:
        return 1
    score = 0.0
    for c in abn:
        score += base_weight / max(train_freq.get(c, 1), 1)
    focus_boost = 1.6 if any(c in FOCUS_CLASS_IDS for c in abn) else 1.0
    rare_boost = 1.25 if any(c in RARE_CLASS_IDS for c in abn) else 1.0
    return int(np.clip(round(score * focus_boost * rare_boost), 1, 4))

print("Helpers ready")

Helpers ready


# Cell 6 — Build manifests from stage1 bundle paths

In [6]:
def write_manifest(paths, out_txt):
    with open(out_txt, "w") as f:
        f.write("\n".join(paths))
    return out_txt

final_train_paths = []

# base full images
for _, r in train_records_df.iterrows():
    if r["kind"] == "full":
        abs_path = rel_to_abs(r["img_rel_path"])
        final_train_paths.extend([abs_path] * repeat_base(parse_classes(r["classes"])))

# ROI and tight ROI images
for _, r in train_records_df.iterrows():
    if r["kind"] in {"roi", "roi_tight"}:
        abs_path = rel_to_abs(r["img_rel_path"])
        cls = parse_classes(r["classes"])
        focus = any(c in FOCUS_CLASS_IDS for c in cls)
        rare = any(c in RARE_CLASS_IDS for c in cls)

        if r["kind"] == "roi_tight":
            rep = 3 if focus else 1
        else:
            rep = 2 if focus else 1

        if rare:
            rep += 1

        final_train_paths.extend([abs_path] * rep)

# shuffle
rng = np.random.default_rng(SEED)
rng.shuffle(final_train_paths)

val_paths = [rel_to_abs(r["img_rel_path"]) for _, r in val_records_df.iterrows()]

MANIFEST_DIR = os.path.join(WORK_DIR, "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

TRAIN_FINAL_TXT = write_manifest(final_train_paths, os.path.join(MANIFEST_DIR, "train_final.txt"))
VAL_TXT = write_manifest(val_paths, os.path.join(MANIFEST_DIR, "val.txt"))

print("Final train lines:", len(final_train_paths))
print("Val lines:", len(val_paths))
print("Sample train path:", final_train_paths[0])
print("Exists:", os.path.exists(final_train_paths[0]))

Final train lines: 38702
Val lines: 3000
Sample train path: /kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle/data/yolo1024/images/train/full/446dc542bff42f41b0beb691f876105e_full.jpg
Exists: True


# Cell 7 — Create final YAML

In [7]:
FINAL_YAML = os.path.join(WORK_DIR, "yolo1024_stage2.yaml")

with open(FINAL_YAML, "w") as f:
    f.write(f"path: {STAGE1_ROOT}\n")
    f.write(f"train: {TRAIN_FINAL_TXT}\n")
    f.write(f"val: {VAL_TXT}\n")
    f.write("names:\n")
    for i, name in enumerate(CLASS_NAMES):
        f.write(f"  {i}: {name}\n")

print(open(FINAL_YAML).read())

path: /kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle
train: /kaggle/working/yolo1024_stage2_bundle/manifests/train_final.txt
val: /kaggle/working/yolo1024_stage2_bundle/manifests/val.txt
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis



# Cell 8 — Stage1 model load# 

In [8]:
assert os.path.exists(MODEL_NAME), f"Stage1 model not found: {MODEL_NAME}"
stage1_model = YOLO(MODEL_NAME)
print("Stage1 model loaded")

Stage1 model loaded


# Cell 9 — Mine hard negatives from normal images

In [9]:
normal_full = train_records_df[(train_records_df["kind"] == "full") & (train_records_df["has_abnormal"] == 0)].copy()

# Limit to keep runtime controlled
if len(normal_full) > 2500:
    normal_full = normal_full.sample(2500, random_state=SEED).reset_index(drop=True)

hard_negs = []

print("Mining hard negatives...")
for batch_paths in tqdm(list(chunks(normal_full["img_rel_path"].tolist(), 8)), desc="hard negatives"):
    abs_paths = [rel_to_abs(p) for p in batch_paths]
    results = stage1_model.predict(
        source=abs_paths,
        imgsz=YOLO_SIZE,
        conf=0.12,
        iou=0.45,
        batch=8,
        device=DEVICE,
        verbose=False,
    )

    for p, res in zip(abs_paths, results):
        if res.boxes is None or len(res.boxes) == 0:
            continue
        max_conf = float(res.boxes.conf.detach().cpu().numpy().max())
        if max_conf >= 0.12:
            hard_negs.append({"img_path": p, "max_conf": max_conf})

hard_neg_df = pd.DataFrame(hard_negs)
if len(hard_neg_df) > 2000:
    hard_neg_df = hard_neg_df.sort_values("max_conf", ascending=False).head(2000).reset_index(drop=True)

print("Hard negatives:", len(hard_neg_df))

Mining hard negatives...


hard negatives:   0%|          | 0/313 [00:00<?, ?it/s]

Hard negatives: 472


# Cell 10 — Mine hard positives for the difficult classes

In [10]:
focus_mask = train_records_df["classes"].apply(lambda s: any(c in FOCUS_CLASS_IDS for c in parse_classes(s)))
focus_full = train_records_df[(train_records_df["kind"] == "full") & (train_records_df["has_abnormal"] == 1) & (focus_mask)].copy()

if len(focus_full) > 2500:
    focus_full = focus_full.sample(2500, random_state=SEED).reset_index(drop=True)

hard_pos = []

print("Mining hard positives...")
for batch_rows in tqdm(list(chunks(focus_full.to_dict("records"), 8)), desc="hard positives"):
    batch_paths = [rel_to_abs(r["img_rel_path"]) for r in batch_rows]
    results = stage1_model.predict(
        source=batch_paths,
        imgsz=YOLO_SIZE,
        conf=0.12,
        iou=0.45,
        batch=8,
        device=DEVICE,
        verbose=False,
    )

    for row, p, res in zip(batch_rows, batch_paths, results):
        if res.boxes is None or len(res.boxes) == 0:
            hard_pos.append({"img_path": p, "reason": "no_boxes"})
            continue

        max_conf = float(res.boxes.conf.detach().cpu().numpy().max())
        if max_conf < 0.20:
            hard_pos.append({"img_path": p, "reason": f"low_conf_{max_conf:.3f}"})

hard_pos_df = pd.DataFrame(hard_pos)
if len(hard_pos_df) > 2000:
    hard_pos_df = hard_pos_df.head(2000).reset_index(drop=True)

print("Hard positives:", len(hard_pos_df))

Mining hard positives...


hard positives:   0%|          | 0/313 [00:00<?, ?it/s]

Hard positives: 40


# Cell 11 — Add hard negatives and hard positives to final manifest

In [11]:
# Oversample difficult classes more aggressively in stage2
stage2_train_paths = list(final_train_paths)

for _, r in hard_neg_df.iterrows():
    stage2_train_paths.extend([r["img_path"]] * 3)

for _, r in hard_pos_df.iterrows():
    stage2_train_paths.extend([r["img_path"]] * 4)

rng = np.random.default_rng(SEED)
rng.shuffle(stage2_train_paths)

TRAIN_STAGE2_TXT = write_manifest(stage2_train_paths, os.path.join(MANIFEST_DIR, "train_stage2.txt"))

print("Stage2 train lines:", len(stage2_train_paths))

Stage2 train lines: 40278


# Cell 12 — Stage2 training

In [12]:
EPOCHS_STAGE2 = 2

final_model = YOLO(MODEL_NAME)

final_model.train(
    data=FINAL_YAML,
    epochs=EPOCHS_STAGE2,
    imgsz=YOLO_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=False,
    amp=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=2e-4,
    weight_decay=1e-4,
    cos_lr=True,
    warmup_epochs=0.2,
    patience=1,

    # very conservative medical augmentations
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=2.0,
    translate=0.02,
    scale=0.05,
    fliplr=0.5,
    mosaic=0.10,
    mixup=0.0,
    close_mosaic=1,

    plots=False,
    save=True,

    project=WORK_DIR,
    name="stage2_final",
    verbose=True,
)

stage2_best = os.path.join(WORK_DIR, "stage2_final", "weights", "best.pt")
print("Stage2 best:", stage2_best)

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=1, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo1024_stage2_bundle/yolo1024_stage2.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle/yolov8m_1024_stage1.pt, momentum=0.937, mosaic=0.1, mu

# Cell 13 — Validate stage2 model

In [13]:
stage2_ckpt = YOLO(stage2_best)
stage2_metrics = stage2_ckpt.val(data=FINAL_YAML, imgsz=YOLO_SIZE, device=DEVICE)
print(stage2_metrics)

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,847,866 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.3 ms, read: 183.1±58.1 MB/s, size: 93.0 KB)
val: Scanning /kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle/data/yolo1024/labels/val/full... 3000 images, 2121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3000/3000 1.2Kit/s 2.5s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle/data/yolo1024/labels/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 188/188 1.3s/it 4:02
                   all       3000       4739      0.332       0.35      0.287      0.147
    Aortic enlargement        638        716      0.659      0.817      0.821      0.513
           Atelectasis         34         41      0.

# Cell 14 — Export final model# 

In [14]:
FINAL_EXPORT = os.path.join(WORK_DIR, "yolov8m_1024_final.pt")
shutil.copy2(stage2_best, FINAL_EXPORT)

summary = {
    "STAGE1_ROOT": STAGE1_ROOT,
    "final_export": FINAL_EXPORT,
    "hard_negatives": int(len(hard_neg_df)),
    "hard_positives": int(len(hard_pos_df)),
    "stage2_train_lines": int(len(stage2_train_paths)),
    "epochs_stage2": int(EPOCHS_STAGE2),
    "focus_classes": sorted(list(FOCUS_CLASS_IDS)),
    "rare_classes": sorted(list(RARE_CLASS_IDS)),
}
with open(os.path.join(WORK_DIR, "stage2_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Final model size (MB):", round(os.path.getsize(FINAL_EXPORT) / (1024 * 1024), 2))

{
  "STAGE1_ROOT": "/kaggle/input/datasets/nareshr22mis0015/yolo1024-stage1-bundle/yolo1024_stage1_bundle",
  "final_export": "/kaggle/working/yolo1024_stage2_bundle/yolov8m_1024_final.pt",
  "hard_negatives": 472,
  "hard_positives": 40,
  "stage2_train_lines": 40278,
  "epochs_stage2": 2,
  "focus_classes": [
    1,
    2,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13
  ],
  "rare_classes": [
    2,
    8,
    9,
    11,
    12,
    13
  ]
}
Final model size (MB): 49.71


In [15]:
import os
from IPython.display import FileLink

# 1. Zip your folder (the fastest compression method)
# Replace 'your_folder_name' with the actual folder you want to download
!zip -r -q output_data.zip /kaggle/working/

# 2. Generate and display the fast download link
FileLink(r'output_data.zip')

/kaggle/working/output_data.zip